# Ingest demo data into MongoDB

Reads `demo_data/bulk_upload.xlsx` (one row per product), attaches the matching image from `demo_data/Imgs/` to each row, and loads everything into `ecom_db.products` with `insert_many`.

Steps:
1. Load the Excel sheet
2. Validate + clean each row with a Pydantic `Product` model
3. Map each row's `image` column to its file and read the bytes
4. `insert_many` into MongoDB
5. Verify (counts + show an image back from the DB)

## 0. Install (only needed once)

In [ ]:
%pip install pandas openpyxl python-dotenv pymongo

## 1. Imports + paths

In [ ]:
import os
import mimetypes
from pathlib import Path
from typing import Literal

import pandas as pd
from bson import Binary
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator
from pymongo import MongoClient

In [ ]:
# find the project root (the folder that contains demo_data), wherever this notebook is run from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "demo_data").exists())

EXCEL_PATH = ROOT / "demo_data" / "bulk_upload.xlsx"
IMG_DIR = ROOT / "demo_data" / "Imgs"

print(EXCEL_PATH, EXCEL_PATH.exists())
print(IMG_DIR, IMG_DIR.exists())

## 2. Load the Excel sheet

In [ ]:
df = pd.read_excel(EXCEL_PATH, sheet_name="Products")

print(df.shape)
df.head()

In [ ]:
# same product name can appear more than once (different price / size / color) -> we keep every row as its own product
df["category"].value_counts()

## 3. Product schema (Pydantic)

Cleans the raw row:
- `category` -> lowercase (`Men` and `men` become the same)
- `size` / `color` -> lists (`"S,M,L"` -> `["S", "M", "L"]`)

In [ ]:
class Product(BaseModel):
    name: str = Field(min_length=1)
    description: str
    price: int = Field(gt=0)
    category: Literal["men", "women", "kids"]
    sizes: list[str]
    colors: list[str]
    image_name: str

    @field_validator("category", mode="before")
    @classmethod
    def lowercase_category(cls, value):
        return str(value).strip().lower()

    @field_validator("sizes", "colors", mode="before")
    @classmethod
    def split_comma_list(cls, value):
        if isinstance(value, str):
            return [item.strip() for item in value.split(",") if item.strip()]
        return value

## 4. Check every row has its image

In [ ]:
missing = [img for img in df["image"] if not (IMG_DIR / img).exists()]

print("rows:", len(df))
print("missing images:", missing)

assert not missing, "fix the missing images before inserting"

## 5. Build the documents (row + image mapped together)

Each document stores the image bytes directly (`bson.Binary`). The images are small (< 300 KB), well under MongoDB's 16 MB document limit.

In [ ]:
def load_image(filename: str) -> dict:
    path = IMG_DIR / filename
    content_type = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    return {
        "filename": filename,
        "content_type": content_type,
        "data": Binary(path.read_bytes()),
    }


documents = []
for row in df.to_dict("records"):
    product = Product(
        name=row["name"],
        description=row["description"],
        price=int(row["price"]),
        category=row["category"],
        sizes=row["size"],
        colors=row["color"],
        image_name=row["image"],
    )
    doc = product.model_dump()
    doc["image"] = load_image(product.image_name)
    documents.append(doc)

print("documents ready:", len(documents))

In [ ]:
# peek at one document (without the raw bytes)
sample = {k: v for k, v in documents[0].items() if k != "image"}
sample["image"] = {k: v for k, v in documents[0]["image"].items() if k != "data"}
sample

## 6. Connect to MongoDB (URI comes from `.env`)

In [ ]:
load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))
client.admin.command("ping")
print("Connected to MongoDB ✅")

db = client["ecom_db"]
collection = db["products"]

## 7. INSERT MANY

`RESET = True` clears the collection first, so re-running this notebook doesn't create duplicates.

In [ ]:
RESET = True

if RESET:
    deleted = collection.delete_many({})
    print("old products deleted:", deleted.deleted_count)

result = collection.insert_many(documents)
print("Inserted ✅", len(result.inserted_ids), "products")

## 8. Verify

In [ ]:
print("total:", collection.count_documents({}))

for cat in ["men", "women", "kids"]:
    print(cat, "->", collection.count_documents({"category": cat}))

In [ ]:
# read without the image bytes (projection) - this is how you'd list products
for p in collection.find({"category": "women", "price": {"$lt": 2000}}, {"image.data": 0}).limit(5):
    print(p["name"], "|", p["price"], "|", p["sizes"], "|", p["image"]["filename"])

In [ ]:
# pull one image back out of MongoDB and show it
from IPython.display import Image, display

p = collection.find_one({"name": "Classic Black Tee"})
print(p["name"], "-", p["price"])
display(Image(data=bytes(p["image"]["data"]), width=250))